In [1]:

import pandas as pd
import pickle

from typing import List
from fastf1.core import Session

import shared_libraries.data_processing as processing

import matplotlib.pyplot as plt
import os

In [2]:
with open("./ig_sessions.pickle", "rb") as file:
    sessions: List[Session] = pickle.load(file)

In [3]:
compound_map = pd.read_csv("./ig_compounds_map.csv")

First, let's get our lap data for all sessions

In [4]:
data_by_circuit = processing.get_data_by_circuit(sessions, compound_map)


Next, we remove laps during which special compounds were used, as compounds other than *SOFT*, *MEDIUM* and *HARD* are outside the scope of this experiment.

Also, we remove laps during which track status was not *green* or *yellow*. That way, we get rid of laps affected by unexpected events. These laps introduce noise and anomalies into the data, which is why we remove them.

In [5]:
for circuit, df in data_by_circuit.items():
    processing.remove_special_compounds(df)
    processing.remove_laps_affected_by_unexpected_events(df)


## Outliers
Below, we look at lap time distributions and outliers for each circuit. Figures are saved to the "figures/" folder.

Laps with pit stops and without pit stops are analyzed separately, as a pit stop greatly affects lap time. 

There are some outliers both among pit stop laps and laps without a pit stop. Those outliers will be removed.

In [6]:
# Creating and saving figures
for circuit, df in data_by_circuit.items():
    with_pit = df[df["IsPitLap"]]
    without_pit = df[(df["IsPitLap"]).__neg__()]

    path = f"./figures/lap_times"
    os.makedirs(path, exist_ok=True)

    figures_and_axes = {}
    figures_and_axes["with_pit"] = {}
    figures_and_axes["without_pit"] = {}

    figures_and_axes["with_pit"]["histogram"] = plt.subplots()
    figures_and_axes["with_pit"]["box_plot"] = plt.subplots()
    figures_and_axes["without_pit"]["histogram"] = plt.subplots()
    figures_and_axes["without_pit"]["box_plot"] = plt.subplots()

    dataframe_parts = {
        "with_pit": with_pit,
        "without_pit": without_pit
    }

    for part_key, plot_type_and_figs in figures_and_axes.items():
        for plot_type, (fig, ax) in plot_type_and_figs.items():
            if plot_type == "histogram":
                ax.set_xlabel("Lap time (s)")
                ax.hist(dataframe_parts[part_key]["LapTimeSeconds"])
            elif plot_type == "box_plot":
                ax.set_ylabel("Lap time (s)")
                ax.boxplot(dataframe_parts[part_key]["LapTimeSeconds"])
            ax.set_title(f"{'Laps without pit stop' if part_key == 'without_pit' else 'Pit stop laps'} — {circuit}")
            fig.savefig(f"{path}/{circuit}_{part_key}_{plot_type}")
            plt.close(fig)


In [7]:
# Removing outliers
for circuit, df in data_by_circuit.items():
    with_pit = df.where(df["IsPitLap"])
    without_pit = df.where(df["IsPitLap"].__neg__())
    processing.remove_outliers(with_pit)
    processing.remove_outliers(without_pit)
    data_by_circuit[circuit] = pd.concat([with_pit, without_pit], axis="index", ignore_index=True)

## Preparing data for ML
In this stage, we process the data further to make it suitable for regression tasks. The following operations are performed:
1. Removal of missing values.
1. Addition of a "RealCompound" column which shows what compound was used during a lap. This is necessary because for every session, different compounds (*C1*, *C2*, ..., *C5*) are chosen for *SOFT*, *MEDIUM* and *HARD* tyres.
1. Conversion of numeric WindDirection values to categorical values such as *N*, *S* or *NW*.
1. Removal of columns that were not selected as features for regression.
1. One-hot encoding of categorical columns.

In [9]:
# Preparing data for ML
for circuit, df in data_by_circuit.items():
    processing.remove_missing_values(df)
    df.reset_index(inplace=True)
    df = df.convert_dtypes()
    processing.add_real_compound(df)
    processing.make_wind_direction_categorical(df)
    processing.remove_columns_for_ml(df)
    df = pd.get_dummies(df)
    data_by_circuit[circuit] = df

In [11]:
with open("ig_data/data_by_circuit", "wb") as file:
    pickle.dump(data_by_circuit, file)